# 🎙️ VoiceTyper: Audio Denoising & Noise Removal Prototype

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burubur/voicetyper/blob/main/notebooks/noise_reduction_prototype.ipynb)

This prototype evaluates and compares **3 speech enhancement & noise reduction techniques** on raw `.wav` recordings from VoiceTyper:
1. **Classical DSP**: 80Hz High-Pass Filter + Spectral Gating (removes AC hum, mic rumble, background hiss)
2. **Neural Voice Activity Detection (Silero VAD)**: Isolates pure vocal segments and discards background noise pauses
3. **Whisper Transcription Comparison**: Evaluates Word Error Rate and hallucination reduction on cleaned audio

In [ ]:
# 1. Install dependencies
!pip install -q noisereduce scipy soundfile matplotlib librosa openai-whisper torch torchaudio

In [ ]:
# 2. Load Audio File (Upload your VoiceTyper .wav file or generate synthetic demo audio)
import io
import soundfile as sf
import librosa
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

try:
    from google.colab import files
    print("Upload a VoiceTyper .wav file (e.g. from ~/.voicetyper/conversation/):")
    uploaded = files.upload()
    if uploaded:
        audio_path = list(uploaded.keys())[0]
        raw_audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    else:
        raise ValueError("No file uploaded, generating demo audio.")
except Exception as e:
    print(f"Note: {e}\nGenerating synthetic demo speech with background fan/hum noise...")
    sr = 16000
    t = np.linspace(0, 3, 3 * sr)
    # Synthetic voice harmonic + 60Hz hum + white noise
    speech = 0.5 * np.sin(2 * np.pi * 220 * t) * (np.sin(2 * np.pi * 2 * t) > 0)
    hum = 0.15 * np.sin(2 * np.pi * 60 * t)
    noise = 0.08 * np.random.normal(0, 1, len(t))
    raw_audio = (speech + hum + noise).astype(np.float32)
    audio_path = "demo_audio.wav"
    sf.write(audio_path, raw_audio, sr)

print(f"✓ Loaded: {audio_path} ({len(raw_audio)/sr:.2f}s @ {sr}Hz)")
print("🎧 Original Audio:")
ipd.display(ipd.Audio(raw_audio, rate=sr))

In [ ]:
# 3. Method 1: Classical DSP (High-Pass + Stationary Spectral Subtraction)
from scipy import signal
import noisereduce as nr

def apply_dsp_denoising(audio, sr=16000):
    # Step A: 80Hz High-Pass Butterworth filter (cuts 50/60Hz hum & low-frequency desk vibrations)
    sos = signal.butter(4, 80, btype='highpass', fs=sr, output='sos')
    filtered = signal.sosfilt(sos, audio)
    
    # Step B: Spectral Subtraction Noise Reduction
    denoised = nr.reduce_noise(
        y=filtered, 
        sr=sr, 
        prop_decrease=0.75, 
        stationary=True,
        n_fft=1024,
        win_length=512,
        hop_length=128
    )
    return denoised

dsp_audio = apply_dsp_denoising(raw_audio, sr)
sf.write("dsp_cleaned.wav", dsp_audio, sr)

print("🎧 Method 1: DSP Cleaned Audio (High-Pass + Spectral Subtraction):")
ipd.display(ipd.Audio(dsp_audio, rate=sr))

In [ ]:
# 4. Method 2: Neural Voice Activity Detection (Silero VAD)
import torch

model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

wav_tensor = torch.from_numpy(raw_audio).float()
speech_timestamps = get_speech_timestamps(wav_tensor, model, sampling_rate=16000, threshold=0.5)

if speech_timestamps:
    vad_audio_tensor = collect_chunks(speech_timestamps, wav_tensor)
    vad_audio = vad_audio_tensor.numpy()
else:
    print("⚠️ No speech detected by VAD!")
    vad_audio = raw_audio

sf.write("vad_cleaned.wav", vad_audio, sr)
print(f"✓ Extracted {len(speech_timestamps)} speech chunks ({len(vad_audio)/sr:.2f}s vs {len(raw_audio)/sr:.2f}s)")
print("🎧 Method 2: Silero VAD (Speech-Only Audio):")
ipd.display(ipd.Audio(vad_audio, rate=sr))

In [ ]:
# 5. Visual Comparison: Waveforms & Spectrograms
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)

def plot_audio_spec(audio, title, row):
    time_axis = np.linspace(0, len(audio)/sr, len(audio))
    axes[row, 0].plot(time_axis, audio, color='#6366f1', alpha=0.8)
    axes[row, 0].set_title(f"{title} - Waveform")
    axes[row, 0].set_ylabel("Amplitude")
    axes[row, 0].grid(True, alpha=0.3)
    
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[row, 1], cmap='magma')
    axes[row, 1].set_title(f"{title} - Spectrogram")
    axes[row, 1].set_ylim(0, 8000)

plot_audio_spec(raw_audio, "1. Original Raw Audio", 0)
plot_audio_spec(dsp_audio, "2. DSP Denoised", 1)
plot_audio_spec(vad_audio, "3. Silero VAD Speech", 2)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Transcribe with Whisper
import whisper

print("Loading Whisper base.en model...")
whisper_model = whisper.load_model("base.en")

print("\n================ Transcription Comparison ================")
res_raw = whisper_model.transcribe(raw_audio)
print(f"1. Raw Audio  : \"{res_raw['text'].strip()}\"")

res_dsp = whisper_model.transcribe(dsp_audio)
print(f"2. DSP Cleaned: \"{res_dsp['text'].strip()}\"")

res_vad = whisper_model.transcribe(vad_audio)
print(f"3. VAD Cleaned: \"{res_vad['text'].strip()}\"")
print("===========================================================")